# Author name match and merge

Disclaimer: this file is currently using data scraped around Oct-Dec 2025, and might not reflect the most recently updated author lists off either SciVal or OpenAlex. This will be updated with newer data ASAP. 

In [3]:
# import libraries 

import pandas as pd
import numpy as np
import json
import ast
import fuzzywuzzy
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
from datetime import datetime
from tqdm import tqdm 
from collections import defaultdict


c:\Users\tania\AppData\Local\Programs\Python\Python311\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [4]:
# script to aggregate dicts for later 

def merge_year_dict_lists(list1, list2):
    combined = list1 + list2
    merged = defaultdict(lambda: {
        "year": None,
        "works_count": 0,
        "oa_works_count": 0,
        "cited_by_count": 0
    })
    
    for item in combined:
        year = item["year"]
        merged[year]["year"] = year
        merged[year]["works_count"] += item.get("works_count", 0)
        merged[year]["oa_works_count"] += item.get("oa_works_count", 0)
        merged[year]["cited_by_count"] += item.get("cited_by_count", 0)
    
    # return sorted list 
    return sorted(merged.values(), key=lambda x: x["year"])

### 1. Read and clean OpenAlex author data for SFU

For the sake of simplicity, limit only to authors who have at least 1 publication since 2020. 

In [5]:
# read OpenAlex data 
oa_authors_raw = pd.read_csv("sfu_all_authors.csv")

oa_authors_raw.head()

,id,orcid,display_name,display_name_alternatives,works_count,cited_by_count,summary_stats,ids,affiliations,last_known_institutions,topics,topic_share,x_concepts,counts_by_year,works_api_url,updated_date,created_date
0,https://openalex.org/A5114378471,https://orcid.org/0000-0002-0721-8331,B. Liu,"['B. Liu', 'B L Liu', 'B Liu', 'B X Liu', 'B. ...",2266,110309,"{'2yr_mean_citedness': 6.5344311377245505, 'h_...",{'openalex': 'https://openalex.org/A5114378471...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I126520041', 'ro...","[{'id': 'https://openalex.org/T10048', 'displa...","[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '10138342', 'wikidata': 'https://www.w...","[{'year': 1965, 'works_count': 1, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24
1,https://openalex.org/A5019316470,https://orcid.org/0000-0002-7223-2965,M. C. Vetterli,"['M. C. Vetterli', 'A. Vgenopoulos', 'D. Ventu...",1859,105597,"{'2yr_mean_citedness': 5.203389830508475, 'h_i...",{'openalex': 'https://openalex.org/A5019316470...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I1304378839', 'r...","[{'id': 'https://openalex.org/T10048', 'displa...","[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '105702510', 'wikidata': 'https://www....","[{'year': 1983, 'works_count': 1, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24
2,https://openalex.org/A5077377484,https://orcid.org/0000-0003-0027-7969,J. Llorente Merino,"['J. Llorente Merino', 'J Llorente Merino', 'J...",1681,92792,"{'2yr_mean_citedness': 5.543181818181818, 'h_i...",{'openalex': 'https://openalex.org/A5077377484...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...","[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '109214941', 'wikidata': 'https://www....","[{'year': 2008, 'works_count': 1, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24
3,https://openalex.org/A5039614567,https://orcid.org/0000-0002-7807-7484,M. Danninger,"['M. Danninger', 'Danninger, M.', 'Danninger, ...",1430,62257,"{'2yr_mean_citedness': 5.130434782608695, 'h_i...",{'openalex': 'https://openalex.org/A5039614567...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...","[{'id': 'https://openalex.org/T10527', 'displa...","[{'id': '109214941', 'wikidata': 'https://www....","[{'year': 2007, 'works_count': 2, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-01T23:18:31.786418,2016-06-24
4,https://openalex.org/A5100397026,https://orcid.org/0000-0003-1991-119X,Hao Zhang,"['Hao Zhang', 'H. Zhang', 'HAO ZHANG', 'HONG-J...",1370,63147,"{'2yr_mean_citedness': 2.3508771929824563, 'h_...",{'openalex': 'https://openalex.org/A5100397026...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I102345215', 'ro...","[{'id': 'https://openalex.org/T10627', 'displa...","[{'id': 'https://openalex.org/T10719', 'displa...","[{'id': '100082104', 'wikidata': 'https://www....","[{'year': 1993, 'works_count': 2, 'oa_works_co...",https://api.openalex.org/works?filter=author.i...,2025-11-02T23:13:26.583910,2016-06-24


In [6]:
# clean OpenAlex data 

oa_authors_clean = oa_authors_raw

# check last year published and number of publications
oa_authors_clean['counts_clean'] = oa_authors_clean['counts_by_year'].apply(lambda x: ast.literal_eval(x))
oa_authors_clean['latest_year'] = oa_authors_clean['counts_clean'].apply(lambda x: x[-1]['year']) # -1 index grabs last year
oa_authors_clean['pubs_in_latest_yr'] = oa_authors_clean['counts_clean'].apply(lambda x: x[-1]['works_count'])

# get rid of unnecessary cols
oa_authors_clean = oa_authors_clean.drop(columns=['topic_share', 'x_concepts', 'counts_by_year', 'updated_date', 'created_date'])

# save only authors who have published since 2020
oa_authors_old = oa_authors_clean[oa_authors_clean['latest_year'] < 2020].reset_index().drop(columns=['index']) # keep for future reference if necessary 
oa_authors_clean = oa_authors_clean[oa_authors_clean['latest_year'] >= 2020].reset_index().drop(columns=['index']) # update clean table

# clean id col 
oa_authors_clean['id'] = oa_authors_clean['id'].apply(lambda x: x.lstrip('https://openalex.org/'))
oa_authors_clean['orcid'] = oa_authors_clean['orcid'].apply(lambda x: str(x).lstrip('https://orcid.org/'))

# deal with names
oa_authors_clean['name_raw'] = oa_authors_clean['display_name'].apply(lambda x: str(x).replace('.', '').replace("'", "").strip(' ').lower())
oa_authors_clean['last_name'] = oa_authors_clean['name_raw'].apply(lambda x: str(x).split(' ')[-1])
oa_authors_clean['last_init'] = oa_authors_clean['last_name'].apply(lambda x: str(x)[0])
oa_authors_clean['first_names'] = oa_authors_clean['name_raw'].apply(lambda x: str(str(x).split(' ')[0:-1]))
oa_authors_clean['potential_matches'] = [[] for _ in range(len(oa_authors_clean))]

# testing 
print(oa_authors_clean['counts_clean'][1][-1]['year'])
print(oa_authors_clean['latest_year'].unique())
print(oa_authors_clean['pubs_in_latest_yr'].unique())

oa_authors_clean

2025
[2025 2022 2023 2024 2020 2021]
[113  90  64  92  44  27  10   7  22  41 167  78  35  37  12   1  45  14
  24  15  66   8  11   9  13  20  38   3  16   6  67  68   2  29  31   5
  77  17   4 214  18  23  57  70  19  21  61  26]


,id,orcid,display_name,display_name_alternatives,works_count,cited_by_count,summary_stats,ids,affiliations,last_known_institutions,topics,works_api_url,counts_clean,latest_year,pubs_in_latest_yr,name_raw,last_name,last_init,first_names,potential_matches
0,A5114378471,0000-0002-0721-8331,B. Liu,"['B. Liu', 'B L Liu', 'B Liu', 'B X Liu', 'B. ...",2266,110309,"{'2yr_mean_citedness': 6.5344311377245505, 'h_...",{'openalex': 'https://openalex.org/A5114378471...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I126520041', 'ro...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1965, 'works_count': 1, 'oa_works_co...",2025,113,b liu,liu,l,['b'],[]
1,A5019316470,0000-0002-7223-2965,M. C. Vetterli,"['M. C. Vetterli', 'A. Vgenopoulos', 'D. Ventu...",1859,105597,"{'2yr_mean_citedness': 5.203389830508475, 'h_i...",{'openalex': 'https://openalex.org/A5019316470...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I1304378839', 'r...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1983, 'works_count': 1, 'oa_works_co...",2025,90,m c vetterli,vetterli,v,"['m', 'c']",[]
2,A5077377484,0000-0003-0027-7969,J. Llorente Merino,"['J. Llorente Merino', 'J Llorente Merino', 'J...",1681,92792,"{'2yr_mean_citedness': 5.543181818181818, 'h_i...",{'openalex': 'https://openalex.org/A5077377484...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2008, 'works_count': 1, 'oa_works_co...",2025,64,j llorente merino,merino,m,"['j', 'llorente']",[]
3,A5039614567,0000-0002-7807-7484,M. Danninger,"['M. Danninger', 'Danninger, M.', 'Danninger, ...",1430,62257,"{'2yr_mean_citedness': 5.130434782608695, 'h_i...",{'openalex': 'https://openalex.org/A5039614567...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2007, 'works_count': 2, 'oa_works_co...",2025,92,m danninger,danninger,d,['m'],[]
4,A5100397026,0000-0003-1991-119X,Hao Zhang,"['Hao Zhang', 'H. Zhang', 'HAO ZHANG', 'HONG-J...",1370,63147,"{'2yr_mean_citedness': 2.3508771929824563, 'h_...",{'openalex': 'https://openalex.org/A5100397026...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I102345215', 'ro...","[{'id': 'https://openalex.org/T10627', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1993, 'works_count': 2, 'oa_works_co...",2025,44,hao zhang,zhang,z,['hao'],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4953,A5119544262,nan,Dominic Jaworiski,['Dominic Jaworiski'],1,0,"{'2yr_mean_citedness': 0.0, 'h_index': 0, 'i10...",{'openalex': 'https://openalex.org/A5119544262...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10783', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2025, 'works_count': 1, 'oa_works_co...",2025,1,dominic jaworiski,jaworiski,j,['dominic'],[]
4954,A5119564484,0000-0003-4437-8817,Stephen Chengxi Li,['Stephen Chengxi Li'],1,1,"{'2yr_mean_citedness': 1.0, 'h_index': 1, 'i10...",{'openalex': 'https://openalex.org/A5119564484...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T14089', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2025, 'works_count': 1, 'oa_works_co...",2025,1,stephen chengxi li,li,l,"['stephen', 'chengxi']",[]
4955,A5119569131,nan,Nima Abdollahpour,['Nima Abdollahpour'],1,0,"{'2yr_mean_citedness': 0.0, 'h_index': 0, 'i10...",{'openalex': 'h

### first catch and merge all exact matches 
THRESHOLD = 100

In [7]:
# first catch and merge all exact matches 
THRESHOLD = 100

perfect_matches = []
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
filepath = f"./intermediate_results/perfect_name_matches_{timestamp}.csv"

for initial, group in tqdm(oa_authors_clean.groupby('last_init')): 
    names = group['name_raw'].to_list()
    ids = group['id'].to_list()

    for i in range(len(names)):
        for j in range(i+1, len(names)):
            name1 = names[i]
            name2 = names[j]

            similarity_score = fuzz.ratio(name1, name2)

            if(similarity_score >= THRESHOLD):
                perfect_matches.append({
                    'name1': name1,
                    'name2': name2, 
                    'id1': ids[i], 
                    'id2': ids[j], 
                    'similarity_score': similarity_score
                })

perfect_matches = pd.DataFrame(perfect_matches)
perfect_matches.to_csv(filepath)

display(perfect_matches)

100%|██████████| 31/31 [00:17<00:00,  1.79it/s]


,name1,name2,id1,id2,similarity_score
0,s bruce archibald,s bruce archibald,A5075000677,A5108658458,100
1,matthew amy,matthew amy,A5042673679,A5092679926,100
2,ronda arab,ronda arab,A5071668228,A5092700665,100
3,mehryar abbasi,mehryar abbasi,A5031764000,A5012315304,100
4,j c burzynski,j c burzynski,A5039744323,A5032934650,100
...,...,...,...,...,...
1985,yichen yan,yichen yan,A5019810767,A5013869037,100
1986,vera yang,vera yang,A5037286751,A5064894884,100
1987,miao zhang,miao zhang,A5100376451,A5092460727,100
1988,peng zhang,peng zhang,A5100364110,A5101104586,100


In [8]:
len(perfect_matches)

1990

In [9]:
# Build adjacency list
graph = defaultdict(set)

for _, row in perfect_matches.iterrows():
    graph[row['id1']].add(row['id2'])
    graph[row['id2']].add(row['id1'])


visited = set()
groups = []

for node in graph:
    if node not in visited:
        stack = [node]
        component = []

        while stack:
            current = stack.pop()
            if current not in visited:
                visited.add(current)
                component.append(current)
                stack.extend(graph[current] - visited)

        groups.append(component)


In [10]:
# initialize deduplicated data table
oa_authors_deduplicated = oa_authors_clean

test = oa_authors_clean

for i in tqdm(range(len(groups))):

    #print(len(groups[i]))

    ids_test = groups[i]
    #print(ids_test)

    # make baby dataframe of duplicate author
    test = oa_authors_deduplicated[
        oa_authors_deduplicated['id'].isin(ids_test)
    ].reset_index(drop=True)

    # init new row
    new_row = test.iloc[[0]].copy()

    # 1. set id
    new_row['id'] = test.loc[0]['id']

    # loop rest of duplicate ids
    for j in range(1, len(groups[i])):
        # add potential id match
        idx = new_row.index[0]
        new_row.at[idx, 'potential_matches'] = (
            new_row.at[idx, 'potential_matches'] + [test.loc[j, 'id']]
        )

        # 2. resolve oricd
        if pd.notna(test.loc[0]['orcid']):
            new_row['orcid'] = test.loc[0]['orcid']
        elif pd.notna(test.loc[j, 'orcid']):
            new_row['orcid'] = test.loc[j]['orcid']
        else: 
            new_row['orcid'] = np.nan

        # 3. leave display name as is

        # 4. merge alt names 
        new_row['display_name_alternatives'] = new_row['display_name_alternatives'] + test.loc[j]['display_name_alternatives']
        
        # 5. sum works count
        new_row['works_count'] = new_row['works_count'] + test.loc[j]['works_count']
        
        # 6. sum citations count 
        new_row['cited_by_count'] = new_row['cited_by_count'] + test.loc[j]['cited_by_count']
        
        # 7. deal with summary stats (for later, for now keep biggest only)
        
        # 8. deal with ids (also for later)
        
        # 9. affiliations
        new_row['affiliations'] = new_row['affiliations'] + test.loc[j]['affiliations']
        
        # 10. last known inst
        new_row['last_known_institutions'] = new_row['last_known_institutions'] + test.loc[j]['last_known_institutions']
        
        # 11. topics
        new_row['topics'] = new_row['topics'] + test.loc[j]['topics']
        
        # 12. works api
        new_row['works_api_url'] = new_row['works_api_url'] + test.loc[j]['works_api_url']
        
        # 13. counts
        new_row.at[idx, 'counts_clean'] = merge_year_dict_lists(
            new_row.at[idx, 'counts_clean'],
            test.loc[j, 'counts_clean']
        )

        # 14. latest year 
        new_row['latest_year'] = max(new_row['latest_year'][0], test['latest_year'][j])

        # 15. works in latest year
        latest_count = new_row.at[idx, 'counts_clean'][-1]['oa_works_count']
        new_row.at[idx, 'pubs_in_latest_yr'] = latest_count

        # 16. name is same 

        # 17. last name is same 

        # 18. last init is same 

        # 19. first names are same 

        # 20. potential matches dealt with above 

    oa_authors_deduplicated = oa_authors_deduplicated[~oa_authors_deduplicated["id"].isin(ids_test)]
    oa_authors_deduplicated = pd.concat([oa_authors_deduplicated, new_row], ignore_index=True)

100%|██████████| 141/141 [00:01<00:00, 103.10it/s]


In [11]:
oa_authors_deduplicated

,id,orcid,display_name,display_name_alternatives,works_count,cited_by_count,summary_stats,ids,affiliations,last_known_institutions,topics,works_api_url,counts_clean,latest_year,pubs_in_latest_yr,name_raw,last_name,last_init,first_names,potential_matches
0,A5114378471,0000-0002-0721-8331,B. Liu,"['B. Liu', 'B L Liu', 'B Liu', 'B X Liu', 'B. ...",2266,110309,"{'2yr_mean_citedness': 6.5344311377245505, 'h_...",{'openalex': 'https://openalex.org/A5114378471...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I126520041', 'ro...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1965, 'works_count': 1, 'oa_works_co...",2025,113,b liu,liu,l,['b'],[]
1,A5019316470,0000-0002-7223-2965,M. C. Vetterli,"['M. C. Vetterli', 'A. Vgenopoulos', 'D. Ventu...",1859,105597,"{'2yr_mean_citedness': 5.203389830508475, 'h_i...",{'openalex': 'https://openalex.org/A5019316470...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I1304378839', 'r...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1983, 'works_count': 1, 'oa_works_co...",2025,90,m c vetterli,vetterli,v,"['m', 'c']",[]
2,A5077377484,0000-0003-0027-7969,J. Llorente Merino,"['J. Llorente Merino', 'J Llorente Merino', 'J...",1681,92792,"{'2yr_mean_citedness': 5.543181818181818, 'h_i...",{'openalex': 'https://openalex.org/A5077377484...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2008, 'works_count': 1, 'oa_works_co...",2025,64,j llorente merino,merino,m,"['j', 'llorente']",[]
3,A5039614567,0000-0002-7807-7484,M. Danninger,"['M. Danninger', 'Danninger, M.', 'Danninger, ...",1430,62257,"{'2yr_mean_citedness': 5.130434782608695, 'h_i...",{'openalex': 'https://openalex.org/A5039614567...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2007, 'works_count': 2, 'oa_works_co...",2025,92,m danninger,danninger,d,['m'],[]
4,A5100397026,0000-0003-1991-119X,Hao Zhang,"['Hao Zhang', 'H. Zhang', 'HAO ZHANG', 'HONG-J...",1370,63147,"{'2yr_mean_citedness': 2.3508771929824563, 'h_...",{'openalex': 'https://openalex.org/A5100397026...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I102345215', 'ro...","[{'id': 'https://openalex.org/T10627', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1993, 'works_count': 2, 'oa_works_co...",2025,44,hao zhang,zhang,z,['hao'],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4629,A5019810767,0000-0002-0717-3962,Yichen Yan,"['Yichen Yan', 'Y Yan', 'Y. Yan', 'Yi-Chen Yan...",92,2187,"{'2yr_mean_citedness': 13.586206896551724, 'h_...",{'openalex': 'https://openalex.org/A5019810767...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I154099455', 'ro...","[{'id': 'https://openalex.org/T10338', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2004, 'works_count': 2, 'oa_works_co...",2025,5,yichen yan,yan,y,['yichen'],[A5013869037]
4630,A5037286751,nan,Vera Yang,"['Vera Yang']['Vera Yang', 'Yang, Vera']",4,102,"{'2yr_mean_citedness': 3.0, 'h_index': 2, 'i10...",{'openalex': 'https://openalex.org/A5037286751...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10836', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2021, 'works_count': 1, 'oa_works_co...",2023,2,vera yang,yang,y,['vera'],[A5064894884]
4631,A5100376451,0000-0001-6126-6142,Miao Zhang,"['Miao Zhang', 'Miao ZHANG', 'None Miao Zhang'...",68,1119

test to see if the deduplication worked. 

if yes, should say 0 at the bottom of next code block

In [12]:

THRESHOLD = 100

perfect_matches = []
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
#filepath = f"./intermediate_results/perfect_name_matches_{timestamp}.csv"

for initial, group in tqdm(oa_authors_deduplicated.groupby('last_init')): 
    names = group['name_raw'].to_list()
    ids = group['id'].to_list()

    for i in range(len(names)):
        for j in range(i+1, len(names)):
            name1 = names[i]
            name2 = names[j]

            similarity_score = fuzz.ratio(name1, name2)

            if(similarity_score >= THRESHOLD):
                perfect_matches.append({
                    'name1': name1,
                    'name2': name2, 
                    'id1': ids[i], 
                    'id2': ids[j], 
                    'similarity_score': similarity_score
                })

perfect_matches = pd.DataFrame(perfect_matches)
#perfect_matches.to_csv(filepath)

display(len(perfect_matches))

100%|██████████| 31/31 [00:15<00:00,  2.01it/s]


0

#### now merge imperfect matches

test different values of the threshold to see what looks best

In [13]:
# test different thresholds 
THRESHOLDS = [75, 78, 80, 82, 85, 88, 90]

for THRESHOLD in THRESHOLDS:

    matches = []
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
    filepath = f"./intermediate_results/name_matches_threshold_{THRESHOLD}.csv"

    print("testing threshold")
    for initial, group in tqdm(oa_authors_deduplicated.groupby('last_init')): 
        names = group['name_raw'].to_list()
        ids = group['id'].to_list()

        for i in range(len(names)):
            for j in range(i+1, len(names)):
                name1 = names[i]
                name2 = names[j]

                similarity_score = fuzz.ratio(name1, name2)

                if(similarity_score >= THRESHOLD):
                    matches.append({
                        'name1': name1,
                        'name2': name2, 
                        'id1': ids[i], 
                        'id2': ids[j], 
                        'similarity_score': similarity_score
                    })

    matches = pd.DataFrame(matches)
    matches.to_csv(filepath)

print("done!")
#display(matches)

testing threshold


100%|██████████| 31/31 [00:15<00:00,  2.05it/s]


testing threshold


100%|██████████| 31/31 [00:14<00:00,  2.07it/s]


testing threshold


...

testing threshold


100%|██████████| 31/31 [00:15<00:00,  2.06it/s]


testing threshold


100%|██████████| 31/31 [00:15<00:00,  2.06it/s]

done!


### threshold used is 85
See excel sheet for more detail 

In [14]:
matches = pd.read_csv("intermediate_results/name_matches_threshold_85.csv").drop(columns=['Unnamed: 0'])

print(len(matches))

display(matches)

158


,name1,name2,id1,id2,similarity_score
0,ahmed al‐rawi,ahmed al-rawi,A5050245548,A5093769593,92
1,vinícius c azevedo,vinicius c azevedo,A5112368463,A5103260690,94
2,joshua d alampi,josh alampi,A5005530879,A5064096264,85
3,mark a brockman,ma brockman,A5021397351,A5062945775,85
4,christopher beh,christopher buse,A5012256170,A5118970217,90
...,...,...,...,...,...
153,zhilin zhang,zhitian zhang,A5100723116,A5111217709,88
154,z zhang,n zhang,A5009011522,A5061007955,86
155,z zhang,j zhang,A5009011522,A5070372260,86
156,vanja zdjelar,ivana zdjelar,A5025292022,A5114472761,92


In [15]:
# Build adjacency list
graph = defaultdict(set)

for _, row in matches.iterrows():
    graph[row['id1']].add(row['id2'])
    graph[row['id2']].add(row['id1'])


visited = set()
groups = []

for node in graph:
    if node not in visited:
        stack = [node]
        component = []

        while stack:
            current = stack.pop()
            if current not in visited:
                visited.add(current)
                component.append(current)
                stack.extend(graph[current] - visited)

        groups.append(component)


In [16]:
# initialize deduplicated data table
oa_authors_deduplicated_2 = oa_authors_deduplicated

test = oa_authors_deduplicated

for i in tqdm(range(len(groups))):

    #print(len(groups[i]))

    ids_test = groups[i]
    #print(ids_test)

    # make baby dataframe of duplicate author
    test = oa_authors_deduplicated_2[
        oa_authors_deduplicated_2['id'].isin(ids_test)
    ].reset_index(drop=True)

    # init new row
    new_row = test.iloc[[0]].copy()

    # 1. set id
    new_row['id'] = test.loc[0]['id']

    # loop rest of duplicate ids
    for j in range(1, len(groups[i])):
        # add potential id match
        idx = new_row.index[0]
        new_row.at[idx, 'potential_matches'] = (
            new_row.at[idx, 'potential_matches'] + [test.loc[j, 'id']]
        )

        # 2. resolve oricd
        if pd.notna(test.loc[0]['orcid']):
            new_row['orcid'] = test.loc[0]['orcid']
        elif pd.notna(test.loc[j, 'orcid']):
            new_row['orcid'] = test.loc[j]['orcid']
        else: 
            new_row['orcid'] = np.nan

        # 3. leave display name as is

        # 4. merge alt names 
        new_row['display_name_alternatives'] = new_row['display_name_alternatives'] + test.loc[j]['display_name_alternatives']
        
        # 5. sum works count
        new_row['works_count'] = new_row['works_count'] + test.loc[j]['works_count']
        
        # 6. sum citations count 
        new_row['cited_by_count'] = new_row['cited_by_count'] + test.loc[j]['cited_by_count']
        
        # 7. deal with summary stats (for later, for now keep biggest only)
        
        # 8. deal with ids (also for later)
        
        # 9. affiliations
        new_row['affiliations'] = new_row['affiliations'] + test.loc[j]['affiliations']
        
        # 10. last known inst
        new_row['last_known_institutions'] = new_row['last_known_institutions'] + test.loc[j]['last_known_institutions']
        
        # 11. topics
        new_row['topics'] = new_row['topics'] + test.loc[j]['topics']
        
        # 12. works api
        new_row['works_api_url'] = new_row['works_api_url'] + test.loc[j]['works_api_url']
        
        # 13. counts
        new_row.at[idx, 'counts_clean'] = merge_year_dict_lists(
            new_row.at[idx, 'counts_clean'],
            test.loc[j, 'counts_clean']
        )

        # 14. latest year 
        new_row['latest_year'] = max(new_row['latest_year'][0], test['latest_year'][j])

        # 15. works in latest year
        latest_count = new_row.at[idx, 'counts_clean'][-1]['oa_works_count']
        new_row.at[idx, 'pubs_in_latest_yr'] = latest_count

        # 16. keep top name

        # 17. keep top last name

        # 18. keep top last init 

        # 19. keep top first names 

        # 20. potential matches dealt with above 

    oa_authors_deduplicated_2 = oa_authors_deduplicated_2[~oa_authors_deduplicated_2["id"].isin(ids_test)]
    oa_authors_deduplicated_2 = pd.concat([oa_authors_deduplicated_2, new_row], ignore_index=True)

100%|██████████| 144/144 [00:01<00:00, 104.24it/s]


In [17]:
oa_authors_deduplicated_2

,id,orcid,display_name,display_name_alternatives,works_count,cited_by_count,summary_stats,ids,affiliations,last_known_institutions,topics,works_api_url,counts_clean,latest_year,pubs_in_latest_yr,name_raw,last_name,last_init,first_names,potential_matches
0,A5019316470,0000-0002-7223-2965,M. C. Vetterli,"['M. C. Vetterli', 'A. Vgenopoulos', 'D. Ventu...",1859,105597,"{'2yr_mean_citedness': 5.203389830508475, 'h_i...",{'openalex': 'https://openalex.org/A5019316470...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I1304378839', 'r...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1983, 'works_count': 1, 'oa_works_co...",2025,90,m c vetterli,vetterli,v,"['m', 'c']",[]
1,A5077377484,0000-0003-0027-7969,J. Llorente Merino,"['J. Llorente Merino', 'J Llorente Merino', 'J...",1681,92792,"{'2yr_mean_citedness': 5.543181818181818, 'h_i...",{'openalex': 'https://openalex.org/A5077377484...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2008, 'works_count': 1, 'oa_works_co...",2025,64,j llorente merino,merino,m,"['j', 'llorente']",[]
2,A5039614567,0000-0002-7807-7484,M. Danninger,"['M. Danninger', 'Danninger, M.', 'Danninger, ...",1430,62257,"{'2yr_mean_citedness': 5.130434782608695, 'h_i...",{'openalex': 'https://openalex.org/A5039614567...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10048', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2007, 'works_count': 2, 'oa_works_co...",2025,92,m danninger,danninger,d,['m'],[]
3,A5100397026,0000-0003-1991-119X,Hao Zhang,"['Hao Zhang', 'H. Zhang', 'HAO ZHANG', 'HONG-J...",1370,63147,"{'2yr_mean_citedness': 2.3508771929824563, 'h_...",{'openalex': 'https://openalex.org/A5100397026...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I102345215', 'ro...","[{'id': 'https://openalex.org/T10627', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1993, 'works_count': 2, 'oa_works_co...",2025,44,hao zhang,zhang,z,['hao'],[]
4,A5100728059,0000-0003-3394-2208,Steven J.M. Jones,"['Steven J.M. Jones', 'Jones, Steve', 'Jones, ...",1284,241753,"{'2yr_mean_citedness': 2.038793103448276, 'h_i...",{'openalex': 'https://openalex.org/A5100728059...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I1335321130', 'r...","[{'id': 'https://openalex.org/T11287', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1929, 'works_count': 1, 'oa_works_co...",2025,27,steven jm jones,jones,j,"['steven', 'jm']",[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4475,A5101842208,0000-0001-8084-7233,Sen Yang,"['Sen Yang', 'None Sen Yang', 'S. Yang', 'Shan...",36,443,"{'2yr_mean_citedness': 5.625, 'h_index': 10, '...",{'openalex': 'https://openalex.org/A5101842208...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I111599522', 'ro...","[{'id': 'https://openalex.org/T10714', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 2007, 'works_count': 1, 'oa_works_co...",2025,2,sen yang,yang,y,['sen'],[A5100592332]
4476,A5050510999,nan,M. A. Yakovlev,"['M. A. Yakovlev', 'M Yakovlev', 'M. Yakovlev'...",29,106,"{'2yr_mean_citedness': 2.0, 'h_index': 3, 'i10...",{'openalex': 'https://openalex.org/A5050510999...,[{'institution': {'id': 'https://openalex.org/...,"[{'id': 'https://openalex.org/I18014758', 'ror...","[{'id': 'https://openalex.org/T10346', 'displa...",https://api.openalex.org/works?filter=author.i...,"[{'year': 1973, 'works_count': 1, 'oa_works_co...",2025,2,m a yakovlev,yakovlev,y,"['m', 'a']",[A5085376161]
4477,A5100723116,0000-0001-5913-4685,Zhilin Zhang,"['

In [53]:
oa_authors_deduplicated_2[oa_authors_deduplicated_2['last_name']== "alperin"]

,id,orcid,display_name,display_name_alternatives,works_count,cited_by_count,summary_stats,ids,affiliations,last_known_institutions,topics,works_api_url,counts_clean,latest_year,pubs_in_latest_yr,name_raw,last_name,last_init,first_names,potential_matches


### 2. Read and clean SciVal author data for SFU

Again, limit only to authors who have published at least once since 2020.

In [29]:
# read SciVal data
sv_authors_raw = pd.read_excel("SciVal SFU 14-24 full author list.xlsx", engine="openpyxl", skiprows=11)

print(len(sv_authors_raw))
sv_authors_raw

10637


,Name,Scholarly Output,Most recent publication,Citations,Citations per Publication,Field-Weighted Citation Impact,h-index,Output in Top 10% Citation Percentiles (field-weighted),Oldest publication (since 1996),Scopus author ID,Scopus author profile,Primary author affiliation*
0,"Stelzer, Bernd",1095.0,2025.0,58459.0,53.4,3.63,144.0,749.0,2009.0,3.522798e+10,https://www.scopus.com/authid/detail.url?autho...,TRIUMF
1,"Stelzer, Bernd",1095.0,2025.0,58459.0,53.4,3.63,144.0,749.0,2009.0,3.522798e+10,https://www.scopus.com/authid/detail.url?autho...,University of British Columbia
2,"Vetterli, Michel C.",1076.0,2025.0,58260.0,54.1,3.68,137.0,732.0,1996.0,5.638165e+10,https://www.scopus.com/authid/detail.url?autho...,University of British Columbia
3,"Vetterli, Michel C.",1076.0,2025.0,58260.0,54.1,3.68,137.0,732.0,1996.0,5.638165e+10,https://www.scopus.com/authid/detail.url?autho...,TRIUMF
4,"Mori, D.",665.0,2025.0,39801.0,59.9,3.67,95.0,362.0,2015.0,5.677971e+10,https://www.scopus.com/authid/detail.url?autho...,Simon Fraser University
...,...,...,...,...,...,...,...,...,...,...,...,...
10632,"Yan, Yichen",1.0,2025.0,0.0,0.0,0.00,0.0,0.0,2025.0,5.949622e+10,https://www.scopus.com/authid/detail.url?autho...,Simon Fraser University
10633,"Zhang, Lei",1.0,2017.0,13.0,13.0,0.82,1.0,0.0,2017.0,5.964549e+10,https://www.scopus.com/authid/detail.url?autho...,Simon Fraser University
10634,"Lui, Caitlyn",1.0,2025.0,0.0,0.0,0.00,0.0,0.0,2025.0,5.967985e+10,https://www.scopus.com/authid/detail.url?autho...,Simon Fraser University
10635,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
print(len(sv_authors_raw.columns))
sv_authors_raw.columns

12


Index(['Name', 'Scholarly Output', 'Most recent publication', 'Citations',
       'Citations per Publication', 'Field-Weighted Citation Impact',
       'h-index', 'Output in Top 10% Citation Percentiles (field-weighted)',
       'Oldest publication (since 1996)', 'Scopus author ID',
       'Scopus author profile', 'Primary author affiliation*'],
      dtype='object')

In [39]:
# clean scival data

sv_authors_clean = sv_authors_raw

# drop unnecessary cols
sv_authors_clean = sv_authors_clean.drop(columns=['Output in Top 10% Citation Percentiles (field-weighted)', 'Oldest publication (since 1996)'])

# keep only authors who published since 2020
sv_authors_old = sv_authors_clean[sv_authors_clean['Most recent publication'] < 2020].reset_index().drop(columns=['index'])
sv_authors_clean = sv_authors_clean[sv_authors_clean['Most recent publication'] >= 2020].reset_index().drop(columns=['index'])

# clean scholarly output
sv_authors_clean['Scholarly Output'] = sv_authors_clean['Scholarly Output'].apply(lambda x: int(x))

# clean most recent pubs 
sv_authors_clean['Most recent publication'] = sv_authors_clean['Most recent publication'].apply(lambda x: int(x))

# clean citations 
sv_authors_clean['Citations'] = sv_authors_clean['Citations'].apply(lambda x: int(x))

# clean h-index 
sv_authors_clean['h-index'] = sv_authors_clean['h-index'].apply(lambda x: int(x))

# clean h-index 
sv_authors_clean['h-index'] = sv_authors_clean['h-index'].apply(lambda x: int(x))

# clean id
sv_authors_clean['Scopus author ID'] = sv_authors_clean['Scopus author ID'].apply(lambda x: int(x))

# deal with names
sv_authors_clean['last_name'] = sv_authors_clean['Name'].apply(lambda x: str(x).split(',')[0].replace('.', '').replace("'", "").strip(' ').lower())
sv_authors_clean['last_init'] = sv_authors_clean['last_name'].apply(lambda x: str(x)[0].replace('.', '').replace("'", "").strip(' ').lower())
sv_authors_clean['first_names'] = sv_authors_clean['Name'].apply(lambda x: str(x).split(',')[-1].replace('.', '').replace("'", "").strip(' ').lower())
sv_authors_clean['name_raw'] = sv_authors_clean['first_names'] + ' ' + sv_authors_clean['last_name']
sv_authors_clean['potential_matches'] = [[] for _ in range(len(sv_authors_clean))]

sv_authors_clean

,Name,Scholarly Output,Most recent publication,Citations,Citations per Publication,Field-Weighted Citation Impact,h-index,Scopus author ID,Scopus author profile,Primary author affiliation*,last_name,last_init,first_names,name_raw,potential_matches
0,"Stelzer, Bernd",1095,2025,58459,53.4,3.63,144,35227985000,https://www.scopus.com/authid/detail.url?autho...,TRIUMF,stelzer,s,bernd,bernd stelzer,[]
1,"Stelzer, Bernd",1095,2025,58459,53.4,3.63,144,35227985000,https://www.scopus.com/authid/detail.url?autho...,University of British Columbia,stelzer,s,bernd,bernd stelzer,[]
2,"Vetterli, Michel C.",1076,2025,58260,54.1,3.68,137,56381653900,https://www.scopus.com/authid/detail.url?autho...,University of British Columbia,vetterli,v,michel c,michel c vetterli,[]
3,"Vetterli, Michel C.",1076,2025,58260,54.1,3.68,137,56381653900,https://www.scopus.com/authid/detail.url?autho...,TRIUMF,vetterli,v,michel c,michel c vetterli,[]
4,"Mori, D.",665,2025,39801,59.9,3.67,95,56779709600,https://www.scopus.com/authid/detail.url?autho...,Simon Fraser University,mori,m,d,d mori,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6660,"Sobhan, Arshia",1,2024,0,0.0,0.00,0,59233806600,https://www.scopus.com/authid/detail.url?autho...,Simon Fraser University,sobhan,s,arshia,arshia sobhan,[]
6661,"Antunes Miranda, Samuel",1,2024,0,0.0,0.00,0,59252190000,https://www.scopus.com/authid/detail.url?autho...,Simon Fraser University,antunes miranda,a,samuel,samuel antunes miranda,[]
6662,"Lin, Anna",1,2024,0,0.0,0.00,0,59464151500,https://www.scopus.com/authid/detail.url?autho...,Simon Fraser University,lin,l,anna,anna lin,[]
6663,"Yan, Yichen",1,2025,0,0.0,0.00,0,59496217500,https://www.scopus.com/authid/detail.url?autho...,Simon Fraser University,yan,y,yichen,yichen yan,[]


### first catch and merge all exact matches
THRESHOLD = 100 

In [42]:
# first catch and merge all exact matches 
THRESHOLD = 100

perfect_matches = []
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
filepath = f"./intermediate_results/sv_perfect_name_matches_{timestamp}.csv"

for initial, group in tqdm(sv_authors_clean.groupby('last_init')): 
    names = group['name_raw'].to_list()
    ids = group['Scopus author ID'].to_list()

    for i in range(len(names)):
        for j in range(i+1, len(names)):
            name1 = names[i]
            name2 = names[j]

            similarity_score = fuzz.ratio(name1, name2)

            if(similarity_score >= THRESHOLD):
                perfect_matches.append({
                    'name1': name1,
                    'name2': name2, 
                    'id1': ids[i], 
                    'id2': ids[j], 
                    'similarity_score': similarity_score
                })

perfect_matches = pd.DataFrame(perfect_matches)
perfect_matches.to_csv(filepath)

display(perfect_matches)

100%|██████████| 29/29 [00:32<00:00,  1.11s/it]


,name1,name2,id1,id2,similarity_score
0,ryan w allen,ryan w allen,57203292425,8866069700,100
1,m stella atkins,m stella atkins,7102310616,57026369400,100
2,muhammad asjad,muhammad asjad,58754889500,50961023600,100
3,sonia s anand,sonia s anand,7201522674,7201522674,100
4,benjamin anderson,benjamin anderson,57373810700,58528712300,100
...,...,...,...,...,...
191,lin zhang,lin zhang,59814155200,58853381800,100
192,lin zhang,lin zhang,59814155200,58853381800,100
193,lin zhang,lin zhang,59814155200,58853381800,100
194,lin zhang,lin zhang,58853381800,58853381800,100


In [43]:
# Build adjacency list
graph = defaultdict(set)

for _, row in perfect_matches.iterrows():
    graph[row['id1']].add(row['id2'])
    graph[row['id2']].add(row['id1'])


visited = set()
groups = []

for node in graph:
    if node not in visited:
        stack = [node]
        component = []

        while stack:
            current = stack.pop()
            if current not in visited:
                visited.add(current)
                component.append(current)
                stack.extend(graph[current] - visited)

        groups.append(component)

In [47]:
# initialize deduplicated data table
sv_authors_deduplicated = sv_authors_clean

test = sv_authors_clean

for i in tqdm(range(len(groups))):

    #print(len(groups[i]))

    ids_test = groups[i]
    #print(ids_test)

    # make baby dataframe of duplicate author
    test = sv_authors_deduplicated[
        sv_authors_deduplicated['Scopus author ID'].isin(ids_test)
    ].reset_index(drop=True)

    # init new row
    new_row = test.iloc[[0]].copy()

    # 1. set id
    new_row['Scopus author ID'] = test.loc[0]['Scopus author ID']

    # loop rest of duplicate ids
    for j in range(1, len(groups[i])):
        # add potential id match
        idx = new_row.index[0]
        new_row.at[idx, 'potential_matches'] = (
            new_row.at[idx, 'potential_matches'] + [test.loc[j, 'Scopus author ID']]
        )
        
        # 5. sum works count
        new_row['Scholarly Output'] = new_row['Scholarly Output'] + test.loc[j]['Scholarly Output']
        
        # 6. sum citations count 
        new_row['Citations'] = new_row['Citations'] + test.loc[j]['Citations']
        
        # 7. citations per publication
        new_row['Citations per Publication'] = new_row['Citations']/new_row['Scholarly Output']
        
        # 8. deal with fwci later

        # 9. deal with h-index later
        
        # 10. leave author profile for now
        
        # 10. last known inst
        #new_row['Primary author affiliation*'] = [[new_row['Primary author affiliation*']], [test.loc[j]['Primary author affiliation*']]]
        
        # 16. name is same 

        # 17. last name is same 

        # 18. last init is same 

        # 19. first names are same 

        # 20. potential matches dealt with above 

    sv_authors_deduplicated = sv_authors_deduplicated[~sv_authors_deduplicated["Scopus author ID"].isin(ids_test)]
    sv_authors_deduplicated = pd.concat([sv_authors_deduplicated, new_row], ignore_index=True)

100%|██████████| 165/165 [00:00<00:00, 207.68it/s]


test to see if the deduplication worked. 

if yes, should say 0 at the bottom of next code block

In [48]:

THRESHOLD = 100

perfect_matches = []
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
#filepath = f"./intermediate_results/perfect_name_matches_{timestamp}.csv"

for initial, group in tqdm(oa_authors_deduplicated.groupby('last_init')): 
    names = group['name_raw'].to_list()
    ids = group['id'].to_list()

    for i in range(len(names)):
        for j in range(i+1, len(names)):
            name1 = names[i]
            name2 = names[j]

            similarity_score = fuzz.ratio(name1, name2)

            if(similarity_score >= THRESHOLD):
                perfect_matches.append({
                    'name1': name1,
                    'name2': name2, 
                    'id1': ids[i], 
                    'id2': ids[j], 
                    'similarity_score': similarity_score
                })

perfect_matches = pd.DataFrame(perfect_matches)
#perfect_matches.to_csv(filepath)

display(len(perfect_matches))

100%|██████████| 31/31 [00:15<00:00,  2.03it/s]


0

#### now merge imperfect matches

test different values of the threshold to see what looks best

In [49]:
# test different thresholds 
THRESHOLDS = [75, 78, 80, 82, 85, 88, 90]

for THRESHOLD in THRESHOLDS:

    matches = []
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
    filepath = f"./intermediate_results/sv_name_matches_threshold_{THRESHOLD}.csv"

    print("testing threshold")
    for initial, group in tqdm(sv_authors_deduplicated.groupby('last_init')): 
        names = group['name_raw'].to_list()
        ids = group['Scopus author ID'].to_list()

        for i in range(len(names)):
            for j in range(i+1, len(names)):
                name1 = names[i]
                name2 = names[j]

                similarity_score = fuzz.ratio(name1, name2)

                if(similarity_score >= THRESHOLD):
                    matches.append({
                        'name1': name1,
                        'name2': name2, 
                        'id1': ids[i], 
                        'id2': ids[j], 
                        'similarity_score': similarity_score
                    })

    matches = pd.DataFrame(matches)
    matches.to_csv(filepath)

print("done!")
#display(matches)

testing threshold


100%|██████████| 29/29 [00:31<00:00,  1.08s/it]


testing threshold


100%|██████████| 29/29 [00:40<00:00,  1.40s/it]


testing threshold


...

testing threshold


100%|██████████| 29/29 [00:38<00:00,  1.32s/it]


testing threshold


100%|██████████| 29/29 [00:37<00:00,  1.30s/it]

done!
